In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from PIL import Image
from transformers import ViTImageProcessor, ViTForImageClassification
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# --- 1. Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BASE_DIR = '/content/CNN_ViT_Hybrid_Vit_Research_Pneumonia_4_Classification/Train_Test_Split_Vit_Data'
train_data_dir = os.path.join(BASE_DIR, 'train')
val_data_dir = os.path.join(BASE_DIR, 'val')

# --- 2. Initialize ResNet-ViT Hybrid ---
# This specific checkpoint uses a ResNet-50 backbone before the ViT layers
model_name = "google/vit-hybrid-base-bit-384"
feature_extractor_hybrid = ViTImageProcessor.from_pretrained(model_name)

model_resnet_vit = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=4,
    ignore_mismatched_sizes=True
)
model_resnet_vit.to(device)

# --- 3. Dataset Class ---
class PneumoniaDataset(Dataset):
    def __init__(self, data_dir, feature_extractor):
        self.data_dir = data_dir
        self.feature_extractor = feature_extractor
        self.image_paths = []
        self.labels = []
        self.classes = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
        self.class_to_idx = {name: i for i, name in enumerate(self.classes)}

        for class_name in self.classes:
            path = os.path.join(data_dir, class_name)
            for img in os.listdir(path):
                if img.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.image_paths.append(os.path.join(path, img))
                    self.labels.append(self.class_to_idx[class_name])

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        # Processor handles the resizing to 384x384 required by this hybrid model
        inputs = self.feature_extractor(images=image, return_tensors='pt')
        return inputs['pixel_values'].squeeze(0), torch.tensor(self.labels[idx])

# Create Dataloaders
train_dataloader = DataLoader(PneumoniaDataset(train_data_dir, feature_extractor_hybrid), batch_size=16, shuffle=True)
val_dataloader = DataLoader(PneumoniaDataset(val_data_dir, feature_extractor_hybrid), batch_size=16, shuffle=False)

# --- 4. Optimizer ---
optimizer = AdamW(model_resnet_vit.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

In [ ]:
epochs = 25
print(f"Starting ResNet-ViT Training on {device}...")

for epoch in range(epochs):
    model_resnet_vit.train()
    train_loss, correct, total = 0, 0, 0
    for batch in train_dataloader:
        inputs, labels = batch
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model_resnet_vit(inputs).logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    # Validation Phase
    model_resnet_vit.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for v_batch in val_dataloader:
            v_inputs, v_labels = v_batch
            v_inputs, v_labels = v_inputs.to(device), v_labels.to(device)
            val_outputs = model_resnet_vit(v_inputs).logits
            _, v_pred = val_outputs.max(1)
            val_total += v_labels.size(0)
            val_correct += v_pred.eq(v_labels).sum().item()

    print(f"Hybrid Epoch {epoch+1}/{epochs} | Loss: {train_loss/len(train_dataloader):.4f} | "
          f"Train Acc: {100.*correct/total:.2f}% | Val Acc: {100.*val_correct/val_total:.2f}%")

In [ ]:
model_resnet_vit.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_dataloader:
        inputs, labels = batch
        outputs = model_resnet_vit(inputs.to(device)).logits
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# Final Confusion Matrix for your paper
plt.figure(figsize=(10, 8))
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='RdPurp', xticklabels=train_dataloader.dataset.classes, yticklabels=train_dataloader.dataset.classes)
plt.title('ResNet-ViT Hybrid Confusion Matrix')
plt.show()

print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=train_dataloader.dataset.classes))

In [ ]:
import torch.nn.functional as F
from tqdm import tqdm

class RISE_Explainer(nn.Module):
    def __init__(self, model, input_size=(384, 384), gpu_batch=16):
        super().__init__()
        self.model = model
        self.input_size = input_size
        self.gpu_batch = gpu_batch

    def generate_masks(self, N, s, p1):
        """Generates random masks for sampling."""
        cell_size = np.ceil(np.array(self.input_size) / s)
        up_size = (s + 1) * cell_size

        grid = np.random.rand(N, s, s) < p1
        grid = grid.astype('float32')

        masks = np.empty((N, *self.input_size))
        for i in range(N):
            # Bilinear upsampling and random cropping for jitter
            x = np.random.randint(0, cell_size[0])
            y = np.random.randint(0, cell_size[1])
            masks[i] = self._upsample(grid[i], up_size)[x:x+self.input_size[0], y:y+self.input_size[1]]

        return torch.from_numpy(masks).float().to(device)

    def _upsample(self, grid, up_size):
        # Helper for bilinear interpolation
        img = Image.fromarray(grid)
        return np.array(img.resize((int(up_size[1]), int(up_size[0])), resample=Image.BILINEAR))

    def explain(self, x, N=1000, s=8, p1=0.5):
        """Generates the saliency map for a given image x."""
        self.model.eval()
        masks = self.generate_masks(N, s, p1)

        # Multiply image by masks and run through model
        stack = []
        for i in range(0, N, self.gpu_batch):
            m = masks[i:i+self.gpu_batch].unsqueeze(1)
            masked_x = x * m
            with torch.no_grad():
                logits = self.model(masked_x).logits
                stack.append(torch.softmax(logits, dim=1))

        predictions = torch.cat(stack) # (N, num_classes)

        # Weighted sum of masks based on prediction confidence
        sal = torch.matmul(predictions.data.transpose(0, 1), masks.view(N, -1))
        sal = sal.view(-1, *self.input_size)
        return sal / N / p1

In [ ]:
def visualize_granularity(model, image_path, class_names):
    # Load and process image
    orig_img = Image.open(image_path).convert('RGB').resize((384, 384))
    x = feature_extractor_hybrid(images=orig_img, return_tensors='pt')['pixel_values'].to(device)

    explainer = RISE_Explainer(model)
    saliency_maps = explainer.explain(x) # Shape: (4, 384, 384)

    plt.figure(figsize=(20, 5))
    plt.subplot(1, 5, 1)
    plt.imshow(orig_img)
    plt.title("Original X-ray")
    plt.axis('off')

    for i in range(4):
        plt.subplot(1, 5, i+2)
        plt.imshow(orig_img)
        plt.imshow(saliency_maps[i].cpu(), cmap='jet', alpha=0.5)
        plt.title(f"Score for: {class_names[i]}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# Run for a specific pathogen
visualize_granularity(model_resnet_vit, test_image_path, train_dataloader.dataset.classes)